# Разбор и очистка одной статьи Хабра

Посмотрим путь **JSON → HTML → обычный текст → Document → чанки** на первой статье.
Никаких эмбеддингов и запросов к LLM здесь нет.

Выберите Python из `.mamba` и запускайте ячейки сверху вниз.
Файл большой (около 1,8 ГБ), поэтому читаем только первую статью потоковым загрузчиком.

In [1]:
import re
from collections import Counter
from pathlib import Path
from pprint import pprint
import sys

import pandas as pd
from bs4 import BeautifulSoup

# Работает при запуске из first_lab или из third_semestr.
lab_dir = Path.cwd() if Path("chunking.py").exists() else Path.cwd() / "first_lab"
sys.path.insert(0, str(lab_dir))
from chunking import iter_articles, clean_html, load_documents, split_documents

/Users/arsknz/osnov/coursework/pfur-coursework/intro-to-it/third_semestr/first_lab/.mamba/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Как устроен JSON

Верхний уровень — словарь: ключ обозначает хаб, значение — список статей.
Одна статья — тоже словарь, с текстом, автором, статистикой и другими полями.

```text
{
  "programming": [
    {"id": "960974", "titleHtml": "...", "textHtml": "...", ...},
    ...
  ],
  ...
}
```

`iter_articles()` читает JSON через `ijson` по одной статье.
`next()` забирает только первую запись — весь список в память не загружается.

In [2]:
articles = iter_articles()
hub, article = next(articles)
articles.close()

print("Хаб:", hub)
print("ID:", article["id"])
print("Заголовок:", article["titleHtml"])
print("Дата:", article["timePublished"])
print("Полей в статье:", len(article))

Хаб: programming
ID: 960974
Заголовок: Почему физики и программисты мыслят одинаково: законы симметрии кода и Вселенной
Дата: 2025-10-28T13:40:07+00:00
Полей в статье: 34


In [3]:
# Все поля первой статьи: тип и размер значения, без вывода вложенного служебного содержимого.
fields = pd.DataFrame([
    {"поле": key, "тип": type(value).__name__,
    "размер": len(value) if isinstance(value, (str, list, dict)) else None}
    for key, value in article.items()
])
with pd.option_context("display.max_rows", None):
    display(fields)

,поле,тип,размер
0,id,str,6.0
1,timePublished,str,25.0
2,isCorporative,bool,NaN
3,lang,str,2.0
4,titleHtml,str,80.0
5,leadData,dict,4.0
6,editorVersion,str,3.0
7,postType,str,7.0
8,postLabels,list,0.0
9,author,dict,14.0


## 2. Какие поля берём

| Поле | Что делаем |
|---|---|
| `titleHtml` | Очищаем и добавляем заголовок перед текстом |
| `textHtml` | Берём полный текст статьи и очищаем HTML |
| `leadData.textHtml` | Берём только если полного текста нет; иначе вступление может повториться |
| `id`, `timePublished`, ключ хаба | Сохраняем как метаданные, по ID строим ссылку |
| `author`, `statistics`, `relatedData` и остальные служебные поля | В текст для RAG не добавляем |

В данном примере анонс уже присутствует в начале полного текста.

In [4]:
print("Поля leadData:", list(article["leadData"]))
print("\nАнонс:")
print(article["leadData"]["textHtml"])

raw_html = article["textHtml"]
print("\nНачало полного HTML (первые 1800 символов):")
print(raw_html[:1800])

Поля leadData: ['textHtml', 'imageUrl', 'buttonTextHtml', 'image']

Анонс:
<p>Иногда кажется, что физика и программирование живут в разных мирах. Но чем глубже смотришь, тем сильнее понимаешь — законы мышления, которыми мы руководствуемся, удивительно схожи. Симметрия, порядок, борьба с энтропией, поиск инвариантов — всё это объединяет инженеров, физиков и программистов в одном стремлении понять, как устроен мир. Я просто захотел порассуждать об этом и поделиться с вами своими мыслями. </p>

Начало полного HTML (первые 1800 символов):
<div xmlns="http://www.w3.org/1999/xhtml"><p>Иногда кажется, что физика и программирование живут в разных мирах. Но чем глубже смотришь, тем сильнее понимаешь — законы мышления, которыми мы руководствуемся, удивительно схожи. Симметрия, порядок, борьба с энтропией, поиск инвариантов — всё это объединяет инженеров, физиков и программистов в одном стремлении понять, как устроен мир. Я просто захотел порассуждать об этом и поделиться с вами своими мыслями.  

## 3. Что находится внутри HTML

Разберём HTML с BeautifulSoup и посчитаем теги.
Это помогает увидеть абзацы, подзаголовки, картинки и возможные скрипты.
HTML печатаем как текст, не исполняем его в ноутбуке.

In [5]:
soup = BeautifulSoup(raw_html, "html.parser")
Counter(tag.name for tag in soup.find_all())

Counter({'p': 14, 'h4': 6, 'br': 6, 'hr': 5, 'div': 1, 'figure': 1, 'img': 1})

## 4. Очистка по шагам

Сначала удаляем `script`, `style` и `svg` вместе с содержимым.
В этой конкретной статье их может не быть — это общее правило загрузчика.
Затем добавляем переносы около блочных тегов, чтобы абзацы и заголовки не склеились.

In [6]:
removed = soup.find_all(["script", "style", "svg"])
print("Удаляемых элементов:", len(removed))
for tag in removed:
    tag.decompose()

block_tags = ["p", "div", "br", "li", "h1", "h2", "h3", "h4", "pre", "tr"]
for tag in soup.find_all(block_tags):
    tag.insert_before("\n")
    tag.insert_after("\n")

text = soup.get_text()
print(repr(text[:700]))  # repr показывает переносы строк как \n

Удаляемых элементов: 0
'\n\nИногда кажется, что физика и программирование живут в разных мирах. Но чем глубже смотришь, тем сильнее понимаешь — законы мышления, которыми мы руководствуемся, удивительно схожи. Симметрия, порядок, борьба с энтропией, поиск инвариантов — всё это объединяет инженеров, физиков и программистов в одном стремлении понять, как устроен мир. Я просто захотел порассуждать об этом и поделиться с вами своими мыслями.  \n\nКогда Вселенная компилируется\n\nИногда хочется просто остановиться, закрыть IDE и подумать — а почему вообще мы так устроены? Почему логика кода кажется такой естественной, будто она уже где-то была заложена?\n\nЭта мысль не давала мне покоя после одной симуляции маятника, где всё нач'


`get_text()` убирает теги и декодирует HTML-сущности.
Изображения не распознаём: у `<img>` нет текстового содержимого.
Далее заменяем неразрывные пробелы и убираем невидимый символ `\u200b`,
сжимаем лишние пробелы и оставляем не более одной пустой строки подряд.

Слова, регистр и пунктуацию здесь сохраняем. Стоп-слова не удаляем:
полный читаемый текст понадобится эмбеддеру и LLM. Таблицы и отступы кода
упрощаются — это ограничение нашей учебной очистки.

In [7]:
text = text.replace("\xa0", " ").replace("\u200b", "")
text = re.sub(r"[^\S\n]+", " ", text)
text = "\n".join(line.strip() for line in text.splitlines())
cleaned_text = re.sub(r"\n{3,}", "\n\n", text).strip()

print("Символов в HTML:", len(raw_html))
print("Символов после очистки:", len(cleaned_text))
print("Совпадает с clean_html() из chunking.py:", cleaned_text == clean_html(raw_html))
print("\nОЧИЩЕННЫЙ ТЕКСТ:\n")
print(cleaned_text)

Символов в HTML: 4766
Символов после очистки: 4063
Совпадает с clean_html() из chunking.py: True

ОЧИЩЕННЫЙ ТЕКСТ:

Иногда кажется, что физика и программирование живут в разных мирах. Но чем глубже смотришь, тем сильнее понимаешь — законы мышления, которыми мы руководствуемся, удивительно схожи. Симметрия, порядок, борьба с энтропией, поиск инвариантов — всё это объединяет инженеров, физиков и программистов в одном стремлении понять, как устроен мир. Я просто захотел порассуждать об этом и поделиться с вами своими мыслями.

Когда Вселенная компилируется

Иногда хочется просто остановиться, закрыть IDE и подумать — а почему вообще мы так устроены? Почему логика кода кажется такой естественной, будто она уже где-то была заложена?

Эта мысль не давала мне покоя после одной симуляции маятника, где всё начало вести себя не так, как ожидалось. Я вдруг понял: ведь наш код подчиняется тем же законам, что и природа. У него есть энергия, энтропия, предел устойчивости.

И, может быть, поэтому я р

## 5. Собираем Document

Основной загрузчик делает такую же очистку, добавляет заголовок и метаданные.
Для корпуса он также пропускает пустые записи и дубли по ID или хешу
очищенного текста с заголовком. Здесь загружаем одну статью, поэтому
в статистике дублей ожидаемо нет. Исходный JSON не изменяется.

In [8]:
documents, stats = load_documents(limit=1)
document = documents[0]

print("Статистика загрузки:")
pprint(stats)
print("\nМетаданные (они не являются телом статьи):")
pprint(document.metadata)
print("\nНачало page_content: заголовок + очищенный текст\n")
print(document.page_content[:800])

Статистика загрузки:
{'documents': 1, 'duplicates': 0, 'empty': 0, 'lead_only': 0, 'scanned': 1}

Метаданные (они не являются телом статьи):
{'article_id': '960974',
 'date': '2025-10-28T13:40:07+00:00',
 'hub': 'programming',
 'lead_only': False,
 'source': 'https://habr.com/ru/articles/960974/',
 'title': 'Почему физики и программисты мыслят одинаково: законы симметрии '
          'кода и Вселенной'}

Начало page_content: заголовок + очищенный текст

Почему физики и программисты мыслят одинаково: законы симметрии кода и Вселенной

Иногда кажется, что физика и программирование живут в разных мирах. Но чем глубже смотришь, тем сильнее понимаешь — законы мышления, которыми мы руководствуемся, удивительно схожи. Симметрия, порядок, борьба с энтропией, поиск инвариантов — всё это объединяет инженеров, физиков и программистов в одном стремлении понять, как устроен мир. Я просто захотел порассуждать об этом и поделиться с вами своими мыслями.

Когда Вселенная компилируется

Иногда хочется п

## 6. Рекурсивный чанкинг

`split_documents()` использует `RecursiveCharacterTextSplitter` из LangChain.
Размер — **1000 символов**, целевое перекрытие — **150 символов**.
Сплиттер сначала старается разделять по абзацам, затем по строкам и пробелам.
Фактическое перекрытие не обязано быть ровно 150 символов.
Метаданные статьи сохраняются в каждом чанке.

In [ ]:
chunks = split_documents(documents, chunk_size=1000, chunk_overlap=150)
print("Получилось чанков:", len(chunks))
pd.DataFrame([
    {"чанк": i, "символов": len(chunk.page_content),
     "начало в исходном тексте": chunk.metadata["start_index"]}
    for i, chunk in enumerate(chunks, start=1)
])

Получилось чанков: 5


,чанк,символов,начало в исходном тексте
0,1,938,0
1,2,792,940
2,3,889,1734
3,4,941,2587
4,5,615,3530


In [10]:
for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- ЧАНК {i}: {len(chunk.page_content)} символов ---\n")
    print(chunk.page_content)


--- ЧАНК 1: 938 символов ---

Почему физики и программисты мыслят одинаково: законы симметрии кода и Вселенной

Иногда кажется, что физика и программирование живут в разных мирах. Но чем глубже смотришь, тем сильнее понимаешь — законы мышления, которыми мы руководствуемся, удивительно схожи. Симметрия, порядок, борьба с энтропией, поиск инвариантов — всё это объединяет инженеров, физиков и программистов в одном стремлении понять, как устроен мир. Я просто захотел порассуждать об этом и поделиться с вами своими мыслями.

Когда Вселенная компилируется

Иногда хочется просто остановиться, закрыть IDE и подумать — а почему вообще мы так устроены? Почему логика кода кажется такой естественной, будто она уже где-то была заложена?

Эта мысль не давала мне покоя после одной симуляции маятника, где всё начало вести себя не так, как ожидалось. Я вдруг понял: ведь наш код подчиняется тем же законам, что и природа. У него есть энергия, энтропия, предел устойчивости.

--- ЧАНК 2: 792 символов ---


Теперь можно менять `chunk_size` и `chunk_overlap` в ячейке выше и смотреть,
где проходят границы. Эти чанки затем поступят в FAISS и BM25.
В приложении индексируем первые **500** уникальных статей, а на вопрос
возвращаем **10** чанков — это два разных ограничения.